In [1]:
!pip install evaluate rouge-score nltk


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1cf3c4225718989cd1ca88c410815fe4aac9376c795ba3bd487ea4973fa61056
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [2]:
!pip install evaluate

In [3]:
!pip install rouge_score


In [4]:
!git clone https://github.com/salaniz/pycocoevalcap.git
!pip install pycocoevalcap

Cloning into 'pycocoevalcap'...
remote: Enumerating objects: 821, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 821 (delta 4), reused 3 (delta 3), pack-reused 809 (from 2)
Receiving objects: 100% (821/821), 130.06 MiB | 18.44 MiB/s, done.
Resolving deltas: 100% (424/424), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 8.6 MB/s eta 0:00:00


In [5]:
import gc
import torch

# Clean Python memory
gc.collect()

# Clean CUDA memory if available
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("✔️ Cache cleared successfully!")

✔️ Cache cleared successfully!


In [6]:
from google.colab import drive
import sys

drive.mount('/content/drive', force_remount=True)
sys.path.append('/content/drive/MyDrive/Models/ArCapModel')

Mounted at /content/drive


In [10]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import torch.backends.cudnn as cudnn
import torch.optim
import torch.utils.data
import torchvision.transforms as transforms
import torch.nn.functional as F
from tqdm import tqdm
import evaluate
import json
from nltk.translate.bleu_score import corpus_bleu

from the_datasets2 import *
from utils2 import *
from models2 import DecoderWithAttention, Attention


#Parameters

data_folder = "/content/drive/MyDrive/Models/ArCapModel/FinalDataset"
data_name = 'coco_5_cap_per_img_5_min_word_freq'
checkpoint_file ="/content/drive/MyDrive/Models/ArCapModel/Checkpoints_best/coco_5_cap_per_img_5_min_word_freq_checkpoint.pth.tar" #
word_map_file = '/content/drive/MyDrive/Models/ArCapModel/FinalDataset/WORDMAP_coco_5_cap_per_img_5_min_word_freq.json'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cudnn.benchmark = True


#Load Word Map
with open(word_map_file, 'r') as j:
    word_map = json.load(j)
rev_word_map = {v: k for k, v in word_map.items()}
vocab_size = len(word_map)


#Load Model (state_dict only)
torch.serialization.add_safe_globals([DecoderWithAttention, Attention])

checkpoint = torch.load(checkpoint_file, map_location=device, weights_only=False)
print("Loaded checkpoint keys:", list(checkpoint.keys()))

decoder = DecoderWithAttention(
    attention_dim=1024,
    embed_dim=1024,
    decoder_dim=1024,
    vocab_size=len(word_map),
    dropout=0.5
).to(device)

decoder.load_state_dict(checkpoint["decoder_state_dict"])
decoder.eval()
print("Decoder weights loaded correctly and ready for evaluation.")


bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
rouge = evaluate.load("rouge")


def compute_metrics(references, hypotheses):
    results = {}
    results["Bleu_4"] = bleu.compute(predictions=hypotheses, references=[[r] for r in references])["bleu"]
    results["METEOR"] = meteor.compute(predictions=hypotheses, references=[[r] for r in references])["meteor"]
    results["ROUGE_L"] = rouge.compute(predictions=hypotheses, references=[[r] for r in references])["rougeL"]
    return results


#Evaluation Loop
def evaluate_model(beam_size=5):
    loader = torch.utils.data.DataLoader(
        CaptionDataset(data_folder, data_name, 'TEST'),
        batch_size=1, shuffle=True, num_workers=1, pin_memory=torch.cuda.is_available()
    )

    references, hypotheses = [], []
    img_idx_list = []  # جديييييييييييييد
    for i, (image_features, caps, caplens, allcaps, img_idx) in enumerate( # uuuuuuuuuuuuuuuu
       tqdm(loader, desc=f"EVALUATING AT BEAM SIZE {beam_size}")):

        img_idx_list.append(int(img_idx.item())) # جديييييييييييد


        k = beam_size
        image_features = image_features.to(device)
        image_features_mean = image_features.mean(1).expand(k, 2048)

        k_prev_words = torch.LongTensor([[word_map['<start>']]] * k).to(device)
        seqs = k_prev_words
        top_k_scores = torch.zeros(k, 1).to(device)
        complete_seqs, complete_seqs_scores = [], []
        step = 1

        h1, c1 = decoder.init_hidden_state(k)
        h2, c2 = decoder.init_hidden_state(k)

        while True:
            embeddings = decoder.embedding(k_prev_words).squeeze(1)
            h1, c1 = decoder.top_down_attention(
                torch.cat([h2, image_features_mean, embeddings], dim=1), (h1, c1)
            )
            attention_weighted_encoding = decoder.attention(image_features, h1)
            h2, c2 = decoder.language_model(
                torch.cat([attention_weighted_encoding, h1], dim=1), (h2, c2)
            )

            scores = decoder.fc(h2)
            scores = F.log_softmax(scores, dim=1)
            scores = top_k_scores.expand_as(scores) + scores

            if step == 1:
                top_k_scores, top_k_words = scores[0].topk(k, 0, True, True)
            else:
                top_k_scores, top_k_words = scores.view(-1).topk(k, 0, True, True)

            vocab_size_model = decoder.vocab_size
            prev_word_inds = top_k_words // vocab_size_model
            next_word_inds = top_k_words % vocab_size_model

            seqs = torch.cat([seqs[prev_word_inds], next_word_inds.unsqueeze(1)], dim=1)

            incomplete_inds = [
                ind for ind, next_word in enumerate(next_word_inds)
                if next_word != word_map['<end>']
            ]
            complete_inds = list(set(range(len(next_word_inds))) - set(incomplete_inds))

            if len(complete_inds) > 0:
                complete_seqs.extend(seqs[complete_inds].tolist())
                complete_seqs_scores.extend(top_k_scores[complete_inds])
            k -= len(complete_inds)

            if k == 0:
                break
            seqs = seqs[incomplete_inds]
            h1 = h1[prev_word_inds[incomplete_inds]]
            c1 = c1[prev_word_inds[incomplete_inds]]
            h2 = h2[prev_word_inds[incomplete_inds]]
            c2 = c2[prev_word_inds[incomplete_inds]]
            image_features_mean = image_features_mean[prev_word_inds[incomplete_inds]]
            top_k_scores = top_k_scores[incomplete_inds].unsqueeze(1)
            k_prev_words = next_word_inds[incomplete_inds].unsqueeze(1)

            if step > 50:
                break
            step += 1

        if len(complete_seqs_scores) == 0:
            complete_seqs = seqs.tolist()
            complete_seqs_scores = top_k_scores.squeeze(1).tolist()


        i_best = complete_seqs_scores.index(max(complete_seqs_scores))
        seq = complete_seqs[i_best]

        # Reference
        img_caps = allcaps[0].tolist()
        img_captions = list(map(lambda c: [
            rev_word_map[w] for w in c if w not in
            {word_map['<start>'], word_map['<end>'], word_map['<pad>']}
        ], img_caps))
        img_caps_texts = [' '.join(c) for c in img_captions]
        references.append(img_caps_texts)

        # Hypothesis
        hypothesis = [rev_word_map[w] for w in seq if w not in
                      {word_map['<start>'], word_map['<end>'], word_map['<pad>']}]
        hypotheses.append(' '.join(hypothesis))

    # Save all results

        # جدييييد
    results = [{
    "image_index": int(img_idx_list[i]),
    "reference": ref,
    "generated": hyp
    } for i, (ref, hyp) in enumerate(zip(references, hypotheses))]


    #results = [{"reference": ref, "generated": hyp} for ref, hyp in zip(references, hypotheses)]
    output_path = "/content/drive/MyDrive/Models/ArCapModel/eval_results_M2.json"#remove _best
    with open(output_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to: {output_path}")

    # Compute metrics
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider

    gts = {i: refs for i, refs in enumerate(references)}
    res = {i: [hyp] for i, hyp in enumerate(hypotheses)}

    bleu_scorer = Bleu(4)
    bleu_score, _ = bleu_scorer.compute_score(gts, res)

    meteor_scorer = Meteor()
    meteor_score, _ = meteor_scorer.compute_score(gts, res)

    rouge_scorer = Rouge()
    rouge_score, _ = rouge_scorer.compute_score(gts, res)

    cider_scorer = Cider()
    cider_score, _ = cider_scorer.compute_score(gts, res)

    print("\n Evaluation Metrics:")
    print(f"BLEU-1: {bleu_score[0]:.4f}")
    print(f"BLEU-2: {bleu_score[1]:.4f}")
    print(f"BLEU-3: {bleu_score[2]:.4f}")
    print(f"BLEU-4: {bleu_score[3]:.4f}")
    print(f"METEOR: {meteor_score:.4f}")
    print(f"ROUGE-L: {rouge_score:.4f}")
    print(f"CIDEr:  {cider_score:.4f}")

    return {
        "BLEU": bleu_score,
        "METEOR": meteor_score,
        "ROUGE_L": rouge_score,
        "CIDEr": cider_score
    }

#Run Evaluation
if __name__ == '__main__':
    metrics_dict = evaluate_model(beam_size=5)
    print(metrics_dict)


Loaded checkpoint keys: ['epoch', 'epochs_since_improvement', 'bleu-4', 'decoder', 'decoder_state_dict', 'decoder_optimizer_state_dict']


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Decoder weights loaded correctly and ready for evaluation.


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
EVALUATING AT BEAM SIZE 5: 100%|██████████| 22455/22455 [17:50<00:00, 20.98it/s]


Results saved to: /content/drive/MyDrive/Models/ArCapModel/eval_results_M2.json
{'testlen': 150520, 'reflen': 155530, 'guess': [150520, 128065, 105610, 83155], 'correct': [101435, 51470, 24805, 11170]}
ratio: 0.9677875650999745

 Evaluation Metrics:
BLEU-1: 0.6518
BLEU-2: 0.5034
BLEU-3: 0.3861
BLEU-4: 0.2941
METEOR: 0.3586
ROUGE-L: 0.4855
CIDEr:  0.8774
{'BLEU': [0.6518359365277048, 0.5033885642781276, 0.38612553090547386, 0.29408559966863024], 'METEOR': 0.35861720312742473, 'ROUGE_L': np.float64(0.485458532704934), 'CIDEr': np.float64(0.8773728119148128)}


In [11]:
import os, json, pickle

results_path = "/content/drive/MyDrive/Models/ArCapModel/eval_results_M2.json"
pkl_path = "/content/drive/MyDrive/COCO/Features/all_ids.pkl"
output_path = "/content/drive/MyDrive/Models/ArCapModel/eval_results_M2_with_ids.json"

if os.path.exists(results_path):
    print("File found:", results_path)

    # --- Load JSON results ---
    with open(results_path, 'r') as f:
        data = json.load(f)

    # --- Load mapping image_id -> index ---
    with open(pkl_path, "rb") as f:
        imageid_to_index = pickle.load(f)

    # --- Invert mapping to get index -> image_id ---
    index_to_imageid = {v: k for k, v in imageid_to_index.items()}

    # --- Replace image_index with image_id in all items ---
    for item in data:
        img_idx = item.get("image_index")
        img_id = index_to_imageid.get(img_idx, "UNKNOWN")
        item["image_id"] = img_id        # أضف/استبدل بالـ image_id
        item.pop("image_index", None)    # احذف الـ index القديم

    # --- Save new JSON file ---
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Updated JSON saved to: {output_path}")
    print(f"Total captions saved: {len(data)}")
    print("Showing first samples with image IDs:\n")

    # --- Print first 50 samples ---
    for i, item in enumerate(data[:50]):
        print(f"Sample {i+1} | Image ID: {item['image_id']}")
        print("References:")
        for j, ref in enumerate(item['reference']):
            print(f"  Reference {j+1}: {ref}")

        print("Generated:")
        print(f"  {item['generated']}")
        print("-" * 40)

else:
    print("File not found")


File found: /content/drive/MyDrive/Models/ArCapModel/eval_results_M2.json
Updated JSON saved to: /content/drive/MyDrive/Models/ArCapModel/eval_results_M2_with_ids.json
Total captions saved: 22455
Showing first samples with image IDs:

Sample 1 | Image ID: 281409
References:
  Reference 1: الحكم و الملتقط و الضارب <unk> ان اتم الضارب تارجحه
  Reference 2: يضرب لاعب البيسبول بمضربه بينما ينتظر الحكم و الحكم المساعد خلفه
  Reference 3: مباراه كره القاعده مع ضرب اللاعب للكره و حضور جماهيري كبير في الملعب
  Reference 4: لاعب بيسبول في الدوري الرئيسي يلوح بالمضرب لضرب الكره
  Reference 5: رجل يلوح بمضرب البيسبول بينما يراقبه اخر
Generated:
  لاعب كره القاعده يستعد لضرب الكره
----------------------------------------
Sample 2 | Image ID: 68745
References:
  Reference 1: سله نزهه مفتوحه تحتوي علي ملعقه شوكه سكين طبق و كوب
  Reference 2: سله نزهه من الخوص تحتوي علي ثلاثه دمي دببه
  Reference 3: صوره مقربه لحيوان محشو بجانب كتاب
  Reference 4: تم تجهيز سله نزهه تحمل طابع الدب
  Reference 5: سله ن

In [9]:
import json
from collections import Counter

json_path = "/content/drive/MyDrive/Models/ArQModel/eval_results_with_ids.json"

# --- Load JSON file ---
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# --- Extract all image_ids ---
image_ids = [item["image_id"] for item in data]

# --- Count duplicates ---
counter = Counter(image_ids)

# --- Filter only repeated IDs ---
duplicates = {img_id: count for img_id, count in counter.items() if count > 1}

# --- Print summary ---
print(f"Total unique image_ids: {len(counter)}")
print(f"Total duplicated image_ids: {len(duplicates)}\n")

# --- Print first 20 duplicates with counts ---
if duplicates:
    print("Some duplicated image_ids with counts:")
    for i, (img_id, count) in enumerate(duplicates.items()):
        print(f"{i+1}. Image ID: {img_id} | Count: {count}")
        if i >= 19:  # show only first 20
            break
else:
    print("No duplicates found.")


Total unique image_ids: 4491
Total duplicated image_ids: 4491

Some duplicated image_ids with counts:
1. Image ID: 490878 | Count: 5
2. Image ID: 434915 | Count: 5
3. Image ID: 271852 | Count: 5
4. Image ID: 417284 | Count: 5
5. Image ID: 155736 | Count: 5
6. Image ID: 548423 | Count: 5
7. Image ID: 195353 | Count: 5
8. Image ID: 251717 | Count: 5
9. Image ID: 68380 | Count: 5
10. Image ID: 21498 | Count: 5
11. Image ID: 117089 | Count: 5
12. Image ID: 490008 | Count: 5
13. Image ID: 296236 | Count: 5
14. Image ID: 307523 | Count: 5
15. Image ID: 365094 | Count: 5
16. Image ID: 444302 | Count: 5
17. Image ID: 180798 | Count: 5
18. Image ID: 196839 | Count: 5
19. Image ID: 20111 | Count: 5
20. Image ID: 529578 | Count: 5


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, json

results_path = "/content/drive/MyDrive/Models/ArCapModel/eval_results_best.json"

if os.path.exists(results_path):
    print("File found:", results_path)

    with open(results_path, 'r') as f:
        data = json.load(f)

    print(f"Total captions saved: {len(data)}")
    print("Showing first samples:\n")
    for i, item in enumerate(data[:20]):
        print(f"Sample {i+1}")

        print("References:")
        for j, ref in enumerate(item['reference']):
            print(f"  Reference {j+1}: {ref}")

        print("Generated:")
        print(f"  {item['generated']}")

        print("-" * 40)
else:
    print("File not found")


In [ ]:
import os, json

results_path = "/content/drive/MyDrive/Models/ArCapModel/eval_results_best.json"

# --- Load file ---
with open(results_path, "r") as f:
    data = json.load(f)

print("Total items in file:", len(data))

# --- Detect duplicates ---
seen = set()
duplicates = []

for item in data:
    # Signature = references + generated text
    sig = (tuple(item["reference"]), item["generated"])

    if sig in seen:
        duplicates.append(item)
    else:
        seen.add(sig)

print("Number of duplicates found:", len(duplicates))

# Show first duplicates
if duplicates:
    print("\nShowing first 5 duplicates:\n")
    for i, d in enumerate(duplicates[:100]):
        print(f"Duplicate {i+1}:")
        print("References:")
        for r in d["reference"]:
            print("  -", r)
        print("Generated:", d["generated"])
        print("-" * 40)
else:
    print("No duplicates found.")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Models/ArCapModel/eval_results_best.json'